In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import copy

# Создание датасета

Что нужно в датасет?
- начало игры
- выбор звезды
- активация звезды
- дом отдыха
- шаг уничтожения звезды
- шаг оверкилл звезды
- уничтожение звезды
- конец игры

In [3]:
def find_timestamp(line, which='Pygame', next_word='SPEED'):
    find_word = f'{which}_timestamp'
    t = line[line.find(find_word)+len(find_word)+2:line.find(next_word)-2]
    return int(t)

def find_value(line, word, next_word):
    value = line[line.find(word) + len(word):line.find(next_word)]
    return int(value)

def find_position(line, next_word='Pygame'):
    word = 'Position'
    position = line[line.find(word)+len(word)+2:line.find(next_word)-2]
    x = position[1:position.find(',')]
    y = position[position.find(',')+2:-1]
    return (int(x), int(y))

In [4]:
filename = r'..\data\raw\exp\01TG\im_log_game_1.txt'
df_events = []
points = 0

with open(filename, 'r') as file:
    for line in file:
        
        if line.find('Game started') != -1:
            timestamp = find_timestamp(line, which='Pygame', next_word='SPEED')
            df_events.append({'event': 'start_game', 'timestamp': timestamp, 
                                'pos_x': 0, 'pos_y': 0, 
                                'decision': '-', 'earned_points': 0, 'total_points': 0})
        
        elif line.find('star_selected') != -1:
            timestamp = find_timestamp(line, which='Pygame', next_word='SPEED')
            pos = find_position(line)
            df_events.append({'event': 'select_star', 'timestamp': timestamp, 
                                'pos_x': pos[0], 'pos_y': pos[1], 
                                'decision': '-', 'earned_points': 0, 'total_points': points})
        
        elif line.find('star_unselected') != -1:
            timestamp = find_timestamp(line, which='Pygame', next_word='SPEED')
            pos = find_position(line)
            df_events.append({'event': 'unselect_star', 'timestamp': timestamp, 
                                'pos_x': pos[0], 'pos_y': pos[1], 
                                'decision': '-', 'earned_points': 0, 'total_points': points})
        
        elif line.find('star_activated') != -1:
            timestamp = find_timestamp(line, which='Pygame', next_word='SPEED')
            pos = find_position(line, next_word='Clf')
            df_events.append({'event': 'activate_star', 'timestamp': timestamp, 
                                'pos_x': pos[0], 'pos_y': pos[1], 
                                'decision': '-', 'earned_points': 0, 'total_points': points})
        
        elif line.find('holiday_home.') != -1:
            timestamp = find_timestamp(line, which='Pygame', next_word='SPEED')
            pos = find_position(line, next_word='Triangle')
            df_events.append({'event': 'start_holiday_home', 'timestamp': timestamp, 
                                'pos_x': pos[0], 'pos_y': pos[1], 
                                'decision': '-', 'earned_points': 0, 'total_points': points})
        
        elif line.find('holiday_home_end') != -1:
            timestamp = find_timestamp(line, which='Pygame', next_word='SPEED')
            pos = find_position(line)
            df_events.append({'event': 'end_holiday_home', 'timestamp': timestamp, 
                                'pos_x': pos[0], 'pos_y': pos[1], 
                                'decision': '-', 'earned_points': 0, 'total_points': points})
        
        elif line.find('blast_step') != -1:
            timestamp = find_timestamp(line, which='Pygame', next_word='SPEED')
            if line.find('skipped_empty') != -1:
                event = 'skipped_blast_step'
                pos = find_position(line)
                decision = '-'
                earned_points = 0
            else:
                event = line[line.find('blast_step'):line.find('. P')]
                pos = find_position(line, next_word='Clf')
                decision = line[line.find('Decision:')+len('Decision:')+1:line.find('. Stars')]
                earned_points = find_value(line, word='Score_change: ', next_word='. Pygame')
            points += earned_points
            df_events.append({'event': event, 'timestamp': timestamp, 
                                'pos_x': pos[0], 'pos_y': pos[1], 
                                'decision': decision,'earned_points': earned_points, 'total_points': points})
        elif line.find('overkill_step') != -1:
                timestamp = find_timestamp(line, which='Pygame', next_word='SPEED')
                pos = find_position(line, next_word='Clf')
                
                if line.find('no overkill') != -1:
                    decision = 'no_overkill'
                    earned_points = 0
                else:
                    decision = 'overkill'
                    earned_points = find_value(line, word='Score_change: ', next_word='. Pygame')
                points += earned_points
                df_events.append({'event': 'overkill_step', 'timestamp': timestamp, 
                                  'pos_x': pos[0], 'pos_y': pos[1], 
                                  'decision': decision,'earned_points': earned_points, 'total_points': points})

        elif line.find('score_changed') != -1:
            timestamp = find_timestamp(line, which='Pygame', next_word='SPEED')
            earned_points = int(line[line.find("Earned points")+len('Earned points')+2:line.find('. Total')])
            total_points = line[line.find("Total points")+len('Total points')+2:line.find('. Pygame')]
            #points += earned_points
            df_events.append({'event': 'change_score', 'timestamp': timestamp, 
                                'pos_x': pos[0], 'pos_y': pos[1], 
                                'decision': '-','earned_points': earned_points, 'total_points': points})
        
        elif line.find('Game ended ') != -1:
            timestamp = find_timestamp(line, which='Pygame', next_word='SPEED')
            df_events.append({'event': 'end_game', 'timestamp': timestamp, 
                                'pos_x': 0, 'pos_y': 0, 
                                'decision': '-','earned_points': 0, 'total_points': points})

In [5]:
df_events = pd.DataFrame(df_events)
pause_1_time = df_events.loc[df_events.event == 'start_game']['timestamp'].iloc[0]
df_events['game_timestamp'] = df_events['timestamp'].values - pause_1_time

ind_end_field_1 = df_events.loc[df_events.event == 'end_game'].index[0]
dur_field_2 = df_events.loc[df_events.event == 'end_game']['game_timestamp'].iloc[0]
pause_2_time = df_events.loc[df_events.event == 'start_game']['timestamp'].iloc[1]

df_events.loc[df_events.index > ind_end_field_1, 'game_timestamp'] += (dur_field_2 - (pause_2_time - pause_1_time))
mode = 'im' if filename.find('im_log') != -1 else 'qm'
game = filename[filename.find('log_game')+len('log_game')+1:filename.find('.txt')]
df_events['mode'] = mode
df_events['n_game'] = int(game)
df_events['subject'] = '01TG'
df_events.head(15)

,event,timestamp,pos_x,pos_y,decision,earned_points,total_points,game_timestamp,mode,n_game,subject
0,start_game,2541,0,0,-,0,0,0,im,1,01TG
1,select_star,3049,948,538,-,0,0,508,im,1,01TG
2,activate_star,4553,948,538,-,0,0,2012,im,1,01TG
3,skipped_blast_step,5558,948,538,-,0,0,3017,im,1,01TG
4,blast_step_2,5558,948,538,failure,-15,-15,3017,im,1,01TG
5,blast_step_3,6558,948,538,success,30,15,4017,im,1,01TG
6,blast_step_4,7558,948,538,success,45,60,5017,im,1,01TG
7,blast_step_5,8559,948,538,success,55,115,6018,im,1,01TG
8,blast_step_6,9559,948,538,success,80,195,7018,im,1,01TG
9,blast_step_7,10561,948,538,success,65,260,8020,im,1,01TG


In [5]:
sel_events = df_events.loc[df_events.event == 'activate_star'].game_timestamp.values

star_pos = []
for event in sel_events:
    star_pos.append(df_events.loc[df_events.game_timestamp == event][['pos_x', 'pos_y']].iloc[0])

star_pos = np.unique(np.array(star_pos), axis=0)
n_star = 0
df_events['n_star'] = -1
for pos in star_pos:
    df_events.loc[(df_events.pos_x == pos[0]) & (df_events.pos_y == pos[1]), 'n_star'] = n_star
    n_star += 1

# Подсчёт игровых метрик

## by game

In [27]:
# overkill
n_interaction = 0
n_overkill = 0
n_overkill_all = []
for i in range(len(df_events)-1):
    
    if df_events['event'].iloc[i] == 'change_score':
        n_interaction += 1
        n_overkill_all.append(n_overkill)
        n_overkill = 0

    elif df_events['event'].iloc[i] == 'overkill_step' and df_events['decision'].iloc[i] == 'overkill':
        n_overkill += 1

np.mean(n_overkill_all).round(2)

np.float64(0.55)

In [28]:
n_overkill_all

[0, 0, 0, 1, 0, 0, 0, 2, 0, 2, 0, 0, 1, 2, 0, 1, 1, 0, 0, 1]

In [31]:
df_events.loc[df_events.event == 'overkill_step']

,event,timestamp,pos_x,pos_y,decision,earned_points,total_points,game_timestamp,mode,n_game,subject
11,overkill_step,12564,948,538,no_overkill,0,224,10023,im,1,01TG
23,overkill_step,26159,1254,523,no_overkill,0,416,23618,im,1,01TG
35,overkill_step,38508,576,880,no_overkill,0,589,35967,im,1,01TG
47,overkill_step,50837,1057,710,overkill,-15,786,48296,im,1,01TG
48,overkill_step,51839,1057,710,no_overkill,0,786,49298,im,1,01TG
60,overkill_step,64405,729,734,no_overkill,0,1024,61864,im,1,01TG
76,overkill_step,80636,876,270,no_overkill,0,1307,78095,im,1,01TG
88,overkill_step,94007,636,240,no_overkill,0,1554,91466,im,1,01TG
100,overkill_step,107347,1083,920,overkill,-15,1821,104806,im,1,01TG
101,overkill_step,108347,1083,920,overkill,-9,1812,105806,im,1,01TG


In [ ]:
# holiday_home_number
n_interaction = 0
int_start = False
n_hh = 0
n_hh_all = []
for i in range(len(df_events)-1):
    if df_events['event'].iloc[i] == 'activate_star' and df_events['event'].iloc[i+1].find('step') != -1:
        n_interaction += 1
        n_hh_all.append(n_hh)
        n_hh = 0
        int_start = False
    elif df_events['event'].iloc[i] == 'activate_star' and df_events['event'].iloc[i+1].find('holiday_home') != -1:
        int_start = True
        n_hh += 1
n_hh_av = np.mean(n_hh_all)

In [62]:
n_hh_all, len(n_hh_all), n_interaction

([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0], 20, 20)

In [6]:
# interaction
sel_events = df_events.loc[df_events.event == 'activate_star'].game_timestamp.values

df_steps = []
for i in range(len(sel_events[1:])+1):
    if i != len(sel_events):
        curr_event, next_event = sel_events[i-1], sel_events[i]
    else:
        curr_event, next_event = sel_events[i-1], df_events.game_timestamp.iloc[-1]
    if df_events.loc[df_events.game_timestamp >= curr_event]['event'].iloc[1].find('step') == -1:
        continue
    df_curr = df_events.loc[(df_events.game_timestamp >= curr_event) & (df_events.game_timestamp < next_event)]
    central_steps = range(2, 8)
    if 'skipped_blast_step' in df_curr.event.values:
        central_steps = range(3, 8)
    df_steps.append(df_curr.loc[df_curr.event.isin([f'blast_step_{q}' for q in central_steps])])
df_steps = pd.concat(df_steps, ignore_index=True)

p_green = round(df_steps.loc[df_steps.decision == 'success'].shape[0] / df_steps.shape[0] * 100, 2)

In [77]:
p_green

84.76

In [23]:
df_curr

,event,timestamp,pos_x,pos_y,decision,earned_points,total_points,game_timestamp,mode,n_game,subject
244,activate_star,139208,871,821,-,0,3783,272719,im,1,01TG
245,skipped_blast_step,140212,871,821,-,0,3783,273723,im,1,01TG
246,blast_step_2,140212,871,821,success,5,3788,273723,im,1,01TG
247,blast_step_3,141212,871,821,success,15,3803,274723,im,1,01TG
248,blast_step_4,142216,871,821,failure,-27,3776,275727,im,1,01TG
249,blast_step_5,143218,871,821,failure,-15,3761,276729,im,1,01TG
250,blast_step_6,144220,871,821,failure,-30,3731,277731,im,1,01TG
251,blast_step_7,145221,871,821,failure,-33,3698,278732,im,1,01TG
252,blast_step_8,146226,871,821,success,50,3748,279737,im,1,01TG
253,overkill_step,147229,871,821,no_overkill,0,3748,280740,im,1,01TG


In [26]:
steps = [f'blast_step_{q}' for q in range(1, 9)]
all_steps = df_curr.loc[(df_curr.event.isin(steps)) | (df_curr.event == 'overkill_step') & (df_curr.decision == 'overkill')]
all_steps

,event,timestamp,pos_x,pos_y,decision,earned_points,total_points,game_timestamp,mode,n_game,subject
246,blast_step_2,140212,871,821,success,5,3788,273723,im,1,01TG
247,blast_step_3,141212,871,821,success,15,3803,274723,im,1,01TG
248,blast_step_4,142216,871,821,failure,-27,3776,275727,im,1,01TG
249,blast_step_5,143218,871,821,failure,-15,3761,276729,im,1,01TG
250,blast_step_6,144220,871,821,failure,-30,3731,277731,im,1,01TG
251,blast_step_7,145221,871,821,failure,-33,3698,278732,im,1,01TG
252,blast_step_8,146226,871,821,success,50,3748,279737,im,1,01TG


In [89]:
n_succ = df_events.loc[df_events.decision == 'success'].shape[0]
velocity = float(round(n_succ/df_events.game_timestamp.iloc[-1]*1000*60, 2))
velocity

22.79

In [90]:
n_succ

113

In [92]:
df_events.game_timestamp.iloc[-1]/1000/60

np.float64(4.959149999999999)

In [65]:
df_events.event.value_counts()

event
select_star           25
activate_star         22
change_score          20
blast_step_2          20
blast_step_3          20
blast_step_5          20
blast_step_4          20
blast_step_6          20
blast_step_7          20
blast_step_8          20
skipped_blast_step    10
blast_step_1          10
unselect_star          3
start_game             2
end_game               2
start_holiday_home     2
end_holiday_home       2
Name: count, dtype: int64

## in total

In [30]:
total_score = int(df_events.loc[df_events.event == 'end_game']['total_points'].iloc[1]) # score of the second field in a game
print(f'Absolute score: {total_score} points.')
available_score = int(df_events.loc[df_events.event != 'change_score']['earned_points'].abs().sum())
relative_score = round(total_score / available_score * 100, 2)
print(f'Relative score: {relative_score}%.')

Absolute score: 3873 points.
Relative score: 67.27%.


In [31]:
df_blast_step = df_events.loc[df_events.event.isin([f'blast_step_{i}' for i in range(9)])]
n_success_motor = df_blast_step.loc[df_blast_step.decision == 'success'].shape[0]
df_overkill_step = df_events.loc[df_events.event.isin([f'overkill_step_{i}' for i in range(5)])]
n_success_stop = df_blast_step.loc[df_blast_step.decision == 'no overkill'].shape[0]
p_success = round((n_success_motor + n_success_stop) / (df_blast_step.shape[0] + df_overkill_step.shape[0]) * 100, 2)
print(f'Percent of successful trials: {p_success}%.')

Percent of successful trials: 62.43%.


In [6]:
start_time = df_events.loc[df_events.event == 'start_game']['timestamp'].values
end_time = df_events.loc[df_events.event == 'end_game']['timestamp'].values
game_duration = round(int(sum((end_time - start_time) // 1000)) / 60, 2)
print(f'Game duration: {game_duration} min.')

Game duration: 4.95 min.


In [7]:
bit_rate_min = round(total_score / game_duration, 2)
bit_rate_sec = round(total_score / game_duration / 60, 2)
print(f'Bit rate (amount of points per second): {bit_rate_sec} points/sec.')
print(f'Bit rate (amount of points per minute): {bit_rate_min} points/min.')

Bit rate (amount of points per second): 14.07 points/sec.
Bit rate (amount of points per minute): 844.24 points/min.


In [255]:
n_hh = df_events.loc[df_events.event == 'start_holiday_home'].shape[0]
n_int = df_events.loc[df_events.event == 'activate_star'].shape[0]
H = round(n_hh / n_int * 100, 2)
print(f'N of Interactions = {n_int}, N of Holiday Home = {n_hh}. {H}%.')

N of Interactions = 22, N of Holiday Home = 2. 9.09%.


In [5]:
df_events['event'].value_counts()

event
select_star           25
activate_star         22
blast_step_3          20
blast_step_2          20
change_score          20
blast_step_7          20
blast_step_4          20
blast_step_5          20
blast_step_6          20
overkill_step_1       20
blast_step_8          20
skipped_blast_step    10
blast_step_1          10
overkill_step_2        8
overkill_step_3        3
unselect_star          3
start_game             2
end_game               2
start_holiday_home     2
end_holiday_home       2
Name: count, dtype: int64

## progress

In [8]:
def calculate_score(df):
    total_score = int(df['total_points'].iloc[-1] - df['total_points'].iloc[0])
    available_score = int(df.loc[df.event != 'change_score']['earned_points'].abs().sum())
    relative_score = round(total_score / available_score * 100, 2)
    return relative_score

In [9]:
for min in np.arange(1, game_duration+1):
    score = calculate_score(df_events.loc[(df_events.game_timestamp >= (min-1) * 60 * 1000) & (df_events.game_timestamp < min * 60 * 1000)])
    print(min, score)
print(relative_score)

1.0 70.24
2.0 87.03
3.0 91.7
4.0 69.17
5.0 43.73
76.66


# подсчёт классификационных метрик

In [10]:
def find_proba(line):
    clf_output = line[line.find('[')+1:line.find(']')].split(', ')
    return [float(value) for value in clf_output[-10:]]

In [11]:
filename = r'..\data\raw\exp\01TG\im_log_game_1.txt'

columns_names = ['action'] + [f'act_{i}' for i in range(10)] + [f'step_{i}' for i in range(80)] + [f'overkill_{i}' for i in range(40)] 
df_stars = []
with open(filename, 'r') as file:
    for line in file:

        if line.find('star_activated') != -1:
            overkill_counter = 0
            proba_array = find_proba(line)
        
        elif line.find('holiday_home.') != -1:
            proba_array.extend([float(value) for value in np.full(12*10, np.nan)])
            proba_array.insert(0, 'holiday_home')
            df_stars.append(dict(zip(columns_names, proba_array)))
        
        elif line.find('skipped_empty_blast_step') != -1:
            proba_array.extend([float(value) for value in np.full(10, np.nan)])

        elif line.find(' blast_step') != -1:
            proba_array.extend(find_proba(line))
        
        elif line.find('overkill_step') != -1:
            overkill_counter += 1
            proba_array.extend(find_proba(line))

        elif line.find('star_blasted') != -1:
            if overkill_counter != 4:
                proba_array.extend([float(value) for value in np.full(10*(4-overkill_counter), np.nan)])
            proba_array.insert(0, 'star_blasted')
            df_stars.append(dict(zip(columns_names, proba_array)))
df_stars = pd.DataFrame(df_stars)
df_stars.head()

,action,act_0,act_1,act_2,act_3,act_4,act_5,act_6,act_7,act_8,...,overkill_30,overkill_31,overkill_32,overkill_33,overkill_34,overkill_35,overkill_36,overkill_37,overkill_38,overkill_39
0,star_blasted,0.0329,0.0000,0.0009,0.0003,0.0005,0.0000,0.0002,0.0232,0.1305,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,star_blasted,0.0000,0.0000,0.0005,0.0009,0.0018,0.0013,0.0029,0.0005,0.0177,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,star_blasted,0.0194,0.0001,0.0072,0.0673,0.0598,0.8153,0.4515,0.8744,0.1757,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,star_blasted,0.0000,0.0000,0.0000,0.0000,0.0002,0.0000,0.0000,0.0001,0.0002,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,star_blasted,0.0000,0.0000,0.0085,0.0099,0.0066,0.0903,0.1666,0.1558,0.1838,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
act, steps, overkill = [f'act_{i}' for i in range(10)], [f'step_{i}' for i in range(80)], [f'overkill_{i}' for i in range(40)] 
df_score = []
thr = .5
for star_ind in df_stars.loc[df_stars.action == 'star_blasted'].index:
    score = {}

    star = df_stars.iloc[star_ind]
    n_step1_nan = np.isnan(np.array([float(value) for value in star[steps].values])).sum() # 10 or 0 
    n_steps = len(steps) - n_step1_nan # 70 or 80 
    n_overkill = len(overkill) - np.isnan(np.array([float(value) for value in star[overkill].values])).sum() # 0, 10, 20, 30
    score['A'] = round((sum(star[steps] >= thr) + sum(star[overkill] < thr)) / (n_steps + n_overkill), 2)
    score['A_motor'] = round(sum(star[steps[n_step1_nan+10:-10]] >= thr) / (n_steps-20), 2) # only central ones
    df_score.append(score)
df_score = pd.DataFrame(df_score)

In [22]:
df_score

,A,A_motor
0,0.72,0.80
1,0.61,0.77
2,0.64,0.86
3,0.90,1.00
4,0.75,0.88
5,0.71,0.82
6,0.98,1.00
7,0.75,0.87
8,0.96,0.98
9,0.80,1.00


In [19]:
n_steps

np.int64(70)

In [10]:
def find_latency(inds, queue=5, delay=0):
    inds_diff = np.diff(inds)
    latency = inds[-1]
    for i in range(len(inds_diff[:-queue])):
        if np.array_equal(np.full(queue, 1), inds_diff[i:i+queue]):
            latency = inds[i]
            break
    latency -= delay
    latency *= 100 # in ms
    return latency

def find_max_dur(motor_inds):
    motor_inds_diff = np.diff(motor_inds)
    dur, max_dur = 0, 0
    for shift in motor_inds_diff:
        if shift == 1:
            dur += 1
        else:
            if dur >= max_dur:
                max_dur = copy.deepcopy(dur)
                dur = 0
    if dur >= max_dur:
        max_dur = copy.deepcopy(dur)
    max_dur *= 100
    return max_dur

In [11]:
act, steps, overkill = [f'act_{i}' for i in range(10)], [f'step_{i}' for i in range(80)], [f'overkill_{i}' for i in range(40)] 
thr = 0.5
queue = 3

df_score = []
for star_ind in df_stars.loc[df_stars.action == 'star_blasted'].index:
    score = {}

    star = df_stars.iloc[star_ind]
    n_step1_nan = np.isnan(np.array([float(value) for value in star[steps].values])).sum()
    n_steps = len(steps) - n_step1_nan
    n_overkill = len(overkill) - np.isnan(np.array([float(value) for value in star[overkill].values])).sum()
    score['A'] = round((sum(star[steps] >= thr) + sum(star[overkill] < thr)) / (n_steps + n_overkill), 2)
    score['A_motor'] = round(sum(star[steps] >= thr) / n_steps, 2)

    motor_inds = np.where((star[steps] >= thr) == 1)[0]
    stop_inds = np.where((star[overkill] < thr) == 1)[0]

    score['latency_motor'] = find_latency(motor_inds, queue, n_step1_nan)
    score['latency_stop'] = find_latency(stop_inds, queue)

    score['max_dur'] = find_max_dur(motor_inds)

    df_score.append(score)
df_score = pd.DataFrame(df_score)
df_score['n_star'] = np.arange(20)
df_score.head()

,A,A_motor,latency_motor,latency_stop,max_dur,n_star
0,0.72,0.70,800,0,3800,0
1,0.61,0.57,1200,100,2600,1
2,0.64,0.63,1100,0,4200,2
3,0.90,0.91,700,300,6200,3
4,0.75,0.73,1300,400,4400,4


In [12]:
df_average = df_score.median(numeric_only=True).drop('n_star')
df_average

A                   0.78
A_motor             0.77
latency_motor     800.00
latency_stop      100.00
max_dur          4250.00
dtype: float64

In [227]:
stop_inds

array([ 3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19])

In [13]:
star

action         star_blasted
act_0                0.0001
act_1                0.0254
act_2                0.0013
act_3                0.0053
                   ...     
overkill_35             NaN
overkill_36             NaN
overkill_37             NaN
overkill_38             NaN
overkill_39             NaN
Name: 21, Length: 131, dtype: object